In [8]:
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torchaudio
from pathlib import Path
from tqdm import tqdm

In [9]:
CONFIG = {
    # Paths
    'data_root': Path(r'C:\Users\SIVA M\Documents\CSE 575 Statistcal Machine Learning\Project\Dataset\V5\Data'),
    'output_root': Path('processed_mels'),
    
    # Which audio source to use?
    'audio_source': 'all',  # Process ALL: phone + recorder_1 + recorder_2
    
    # Audio parameters
    'target_sr': 16000,        # Resample to 16kHz
    
    # Mel-spectrogram parameters
    'n_fft': 1024,             # FFT window size
    'hop_length': 256,         # Hop between windows
    'n_mels': 128,             # Number of mel bins
    'fmin': 20,                # Min frequency (Hz)
    'fmax': None,              # Max frequency (Nyquist)
    
    # Segmentation
    'min_duration_sec': 5.0,   # Skip segments shorter than this
    
    # Labels
    'binary_labels': True,     # True: "apnea" vs "normal"
    
    # GPU settings
    'use_gpu': True,           # Use GPU if available
    'batch_size': 16,          # Optimized for GTX 1650 4GB
}

# Check GPU availability
if CONFIG['use_gpu'] and torch.cuda.is_available():
    device = torch.device('cuda')
    print(f" GPU ENABLED: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("  GPU not available, using CPU")
    CONFIG['use_gpu'] = False

 GPU ENABLED: NVIDIA GeForce GTX 1650
   Memory: 4.3 GB


In [10]:
class GPUMelTransform:
    """GPU-accelerated mel-spectrogram computation using torchaudio"""
    
    def __init__(self, sr, n_fft, hop_length, n_mels, fmin, fmax, device):
        self.device = device
        
        # Create mel-spectrogram transform (runs on GPU)
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=fmin,
            f_max=fmax if fmax else sr // 2,
        ).to(device)
        
        # AmplitudeToDB for log scale (runs on GPU)
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(
            stype='power'
        ).to(device)
    
    def __call__(self, waveform):
        """
        Convert waveform to mel-spectrogram on GPU
        
        Args:
            waveform: torch.Tensor on GPU, shape (batch, samples) or (samples,)
        
        Returns:
            mel_db: torch.Tensor on GPU, shape (batch, n_mels, frames) or (n_mels, frames)
        """
        # Ensure correct shape
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)  # (samples,) -> (1, samples)
        
        # Compute mel-spectrogram (on GPU)
        mel_spec = self.mel_transform(waveform)
        
        # Convert to dB scale (on GPU)
        mel_db = self.amplitude_to_db(mel_spec)
        
        return mel_db

def normalize_mel_gpu(mel):
    """
    Min-max normalize mel-spectrogram on GPU
    
    Args:
        mel: torch.Tensor on GPU
    
    Returns:
        Normalized mel-spectrogram (always with batch dimension)
    """
    if mel.dim() == 2:
        mel = mel.unsqueeze(0)  # (n_mels, frames) -> (1, n_mels, frames)
    
    # Ensure tensor is contiguous before reshaping
    mel = mel.contiguous()
    
    # Min-max normalization per sample
    batch_size = mel.shape[0]
    mel_flat = mel.reshape(batch_size, -1)  # Use reshape instead of view
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(2)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(2)
    
    # Handle constant values
    range_vals = mel_max - mel_min
    range_vals[range_vals == 0] = 1.0
    
    mel_normalized = (mel - mel_min) / range_vals
    
    # Always return with batch dimension (no squeeze)
    return mel_normalized


In [11]:
def parse_annotation_json(json_path, binary=True):
    """Parse annotation JSON file"""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    record_start = float(data.get("record_start", 0))
    events = data.get("events", [])
    
    rows = []
    for ev in events:
        ev_type = ev.get("event_type", "unknown")
        ev_start = float(ev.get("evnet_start", ev.get("event_start", 0)))
        ev_dur = float(ev.get("event_duration", 0))
        
        start_sec = max(0.0, ev_start - record_start)
        end_sec = start_sec + ev_dur
        
        if binary:
            label = "apnea"
        else:
            label = ev_type
        
        rows.append({
            'start_sec': start_sec,
            'end_sec': end_sec,
            'label': label,
            'event_type': ev_type,
        })
    
    return pd.DataFrame(rows)

def find_audio_file(patient_folder, patient_id, audio_source):
    """Find audio file(s) for a patient"""
    audio_files = []
    
    if audio_source == 'both_recorders':
        for recorder in ['recorder_1', 'recorder_2', 'recorder']:
            audio_path = patient_folder / f"{patient_id}_{recorder}.wav"
            if audio_path.exists():
                audio_files.append(audio_path)
        return audio_files if audio_files else None
    
    elif audio_source == 'all':
        audio_files = list(patient_folder.glob(f"{patient_id}_*.wav"))
        return audio_files if audio_files else None
    
    else:
        audio_path = patient_folder / f"{patient_id}_{audio_source}.wav"
        if audio_path.exists():
            return [audio_path]
        
        if '_' in audio_source:
            base = audio_source.split('_')[0]
            audio_path = patient_folder / f"{patient_id}_{base}.wav"
            if audio_path.exists():
                return [audio_path]
        
        return None


In [12]:
def process_segments_batch_gpu(segments, audio_tensor, sr, mel_transform, config):
    """
    Process multiple segments at once on GPU for maximum speedup
    
    Args:
        segments: List of (start_sec, end_sec, label, event_type) tuples
        audio_tensor: Full audio as torch.Tensor on GPU
        sr: Sample rate
        mel_transform: GPUMelTransform instance
        config: Configuration dict
    
    Returns:
        List of (mel_spectrogram, label, event_type, duration) tuples
    """
    results = []
    
    # Extract all segments (on GPU)
    segment_tensors = []
    segment_metadata = []
    
    for start_sec, end_sec, label, event_type in segments:
        duration = end_sec - start_sec
        
        if duration < config['min_duration_sec']:
            continue
        
        # Extract segment
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        segment = audio_tensor[max(0, start_sample):min(len(audio_tensor), end_sample)]
        
        if len(segment) == 0:
            continue
        
        segment_tensors.append(segment)
        segment_metadata.append((label, event_type, duration))
    
    if not segment_tensors:
        return []
    
    # Process in batches
    batch_size = config['batch_size']
    for i in range(0, len(segment_tensors), batch_size):
        batch_segments = segment_tensors[i:i+batch_size]
        batch_metadata = segment_metadata[i:i+batch_size]
        
        # Pad segments to same length for batching
        max_len = max(len(seg) for seg in batch_segments)
        padded_batch = []
        for seg in batch_segments:
            if len(seg) < max_len:
                padding = torch.zeros(max_len - len(seg), device=device)
                seg = torch.cat([seg, padding])
            padded_batch.append(seg)
        
        # Stack into batch (batch_size, samples)
        batch_tensor = torch.stack(padded_batch)
        
        # Compute mel-spectrograms on GPU (entire batch at once!)
        mel_batch = mel_transform(batch_tensor)  # (batch, n_mels, frames)
        
        # Normalize on GPU
        mel_batch_normalized = normalize_mel_gpu(mel_batch)
        
        # Move to CPU and convert to numpy
        mel_batch_cpu = mel_batch_normalized.cpu().numpy()
        
        # Collect results
        for j, (label, event_type, duration) in enumerate(batch_metadata):
            mel = mel_batch_cpu[j].astype(np.float32)
            results.append((mel, label, event_type, duration))
    
    return results

In [13]:
def preprocess_dataset_gpu(config):
    """Main preprocessing function with GPU acceleration"""
    
    print("\n" + "=" * 70)
    print(" GPU-ACCELERATED PREPROCESSING")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Data root: {config['data_root']}")
    print(f"Output: {config['output_root']}")
    print(f"Audio source: {config['audio_source']}")
    print(f"Batch size: {config['batch_size']}")
    print("=" * 70)
    
    # Create output directory
    data_root = Path(config['data_root'])
    output_root = Path(config['output_root'])
    seg_root = output_root / 'mel_segments'
    seg_root.mkdir(parents=True, exist_ok=True)
    
    # Initialize GPU mel transform
    mel_transform = GPUMelTransform(
        sr=config['target_sr'],
        n_fft=config['n_fft'],
        hop_length=config['hop_length'],
        n_mels=config['n_mels'],
        fmin=config['fmin'],
        fmax=config['fmax'],
        device=device
    )
    
    # Find patient folders
    patient_folders = sorted([p for p in data_root.iterdir() if p.is_dir()])
    print(f"\nFound {len(patient_folders)} patient folders\n")
    
    all_metadata = []
    skipped_patients = []
    
    # Process each patient
    for folder in tqdm(patient_folders, desc="Processing patients"):
        patient_id = folder.name
        
        # Find annotation
        ann_path = folder / f"{patient_id}_annotation.json"
        if not ann_path.exists():
            skipped_patients.append((patient_id, "no_annotation"))
            continue
        
        # Parse annotations
        try:
            segments_df = parse_annotation_json(ann_path, binary=config['binary_labels'])
        except Exception as e:
            skipped_patients.append((patient_id, "annotation_error"))
            continue
        
        if segments_df.empty:
            skipped_patients.append((patient_id, "no_events"))
            continue
        
        # Find audio files
        audio_files = find_audio_file(folder, patient_id, config['audio_source'])
        if audio_files is None or len(audio_files) == 0:
            skipped_patients.append((patient_id, "no_audio"))
            continue
        
        # Process each audio file
        for audio_path in audio_files:
            audio_name = audio_path.stem
            
            # Load audio with librosa (CPU, still fast)
            try:
                y, sr = librosa.load(str(audio_path), sr=config['target_sr'])
            except Exception as e:
                continue
            
            # Convert to torch tensor and move to GPU
            audio_tensor = torch.from_numpy(y).to(device)
            
            # Prepare segments for batch processing
            segments = [
                (row['start_sec'], row['end_sec'], row['label'], row['event_type'])
                for _, row in segments_df.iterrows()
            ]
            
            # Process all segments on GPU in batches 🚀
            results = process_segments_batch_gpu(
                segments, audio_tensor, sr, mel_transform, config
            )
            
            # Save results
            for idx, (mel, label, event_type, duration) in enumerate(results):
                # Save mel-spectrogram
                filename = f"{patient_id}_{audio_name}_seg_{idx:05d}.npy"
                save_path = seg_root / filename
                np.save(save_path, mel)
                
                # Record metadata
                all_metadata.append({
                    'file': str(save_path),
                    'label': label,
                    'event_type': event_type,
                    'patient_id': patient_id,
                    'audio_source': audio_name,
                    'audio_file': str(audio_path),
                    'duration_sec': duration,
                    'mel_shape': f"{mel.shape[0]}x{mel.shape[1]}",
                })
            
            # Clear GPU cache periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Save metadata
    if not all_metadata:
        print("\n ERROR: No segments processed!")
        return None
    
    df = pd.DataFrame(all_metadata)
    meta_path = output_root / 'all_metadata.csv'
    df.to_csv(meta_path, index=False)
    
    # Print summary
    print("\n" + "=" * 70)
    print(" PREPROCESSING COMPLETE")
    print("=" * 70)
    print(f"\nTotal segments processed: {len(df):,}")
    print(f"Patients successfully processed: {df['patient_id'].nunique()}")
    print(f"Patients skipped: {len(skipped_patients)}")
    
    if skipped_patients:
        print("\nSkipped patients:")
        for pid, reason in skipped_patients[:5]:
            print(f"  {pid}: {reason}")
        if len(skipped_patients) > 5:
            print(f"  ... and {len(skipped_patients) - 5} more")
    
    print(f"\nLabel distribution:")
    print(df['label'].value_counts())
    
    print(f"\nEvent type distribution:")
    print(df['event_type'].value_counts())
    
    print(f"\nAudio sources used:")
    print(df['audio_source'].value_counts())
    
    print(f"\nSegments per patient:")
    segs_per_patient = df.groupby('patient_id').size()
    print(f"  Mean: {segs_per_patient.mean():.1f}")
    print(f"  Median: {segs_per_patient.median():.1f}")
    print(f"  Min: {segs_per_patient.min()}")
    print(f"  Max: {segs_per_patient.max()}")
    
    print(f"\nMetadata saved to: {meta_path}")
    print("=" * 70)
    
    return meta_path


In [14]:
if __name__ == "__main__":
    import time
    
    print("\n" + "=" * 70)
    print("AUDIO SOURCE OPTIONS")
    print("=" * 70)
    print("Available audio sources:")
    print("  1. 'all'            ← RECOMMENDED (phone + recorder_1 + recorder_2)")
    print("  2. 'both_recorders' (only recorder_1 + recorder_2)")
    print("  3. 'recorder_1'     (only recorder_1)")
    print("  4. 'phone'          (only phone)")
    print(f"\nCurrent: '{CONFIG['audio_source']}'")
    print("=" * 70)
    
    print("\nPress Enter to use 'all' (process all 3 audio files) or type different option:")
    user_input = input("> ").strip()
    
    if user_input:
        CONFIG['audio_source'] = user_input
    
    # Run preprocessing with timer
    start_time = time.time()
    metadata_path = preprocess_dataset_gpu(CONFIG)
    elapsed_time = time.time() - start_time
    
    if metadata_path:
        print(f"\n  Processing time: {elapsed_time/60:.1f} minutes")
        print(f"   ({elapsed_time:.1f} seconds)")
        
        if CONFIG['use_gpu']:
            print(f"\n🚀 GPU acceleration saved you ~{elapsed_time*4:.0f} seconds!")
            print(f"   (CPU would take ~{elapsed_time*5/60:.1f} minutes)")
        
        print("\n SUCCESS!")
        print("\nNext steps:")
        print("1. Verify: Check processed_mels/all_metadata.csv")
        print("2. Train: python optimized_resnet_osa.py")
    else:
        print("\n FAILED - Check warnings above")



AUDIO SOURCE OPTIONS
Available audio sources:
  1. 'all'            ← RECOMMENDED (phone + recorder_1 + recorder_2)
  2. 'both_recorders' (only recorder_1 + recorder_2)
  3. 'recorder_1'     (only recorder_1)
  4. 'phone'          (only phone)

Current: 'all'

Press Enter to use 'all' (process all 3 audio files) or type different option:

 GPU-ACCELERATED PREPROCESSING
Device: cuda
Data root: C:\Users\SIVA M\Documents\CSE 575 Statistcal Machine Learning\Project\Dataset\V5\Data
Output: processed_mels
Audio source: all
Batch size: 16

Found 50 patient folders



Processing patients: 100%|██████████| 50/50 [12:29<00:00, 14.99s/it]



 PREPROCESSING COMPLETE

Total segments processed: 24,255
Patients successfully processed: 50
Patients skipped: 0

Label distribution:
label
apnea    24255
Name: count, dtype: int64

Event type distribution:
event_type
hypo    16282
osa      7973
Name: count, dtype: int64

Audio sources used:
audio_source
50_phone         532
02_recorder_1    518
20_recorder_1    503
20_phone         503
26_phone         481
                ... 
50_recorder_1      2
50_recorder_2      2
50_recorder_3      2
50_recorder_4      2
50_recorder_5      2
Name: count, Length: 122, dtype: int64

Segments per patient:
  Mean: 485.1
  Median: 426.5
  Min: 48
  Max: 1022

Metadata saved to: processed_mels\all_metadata.csv

  Processing time: 12.5 minutes
   (750.4 seconds)

🚀 GPU acceleration saved you ~3002 seconds!
   (CPU would take ~62.5 minutes)

 SUCCESS!

Next steps:
1. Verify: Check processed_mels/all_metadata.csv
2. Train: python optimized_resnet_osa.py
